# CellNeighborEX v2 Tutorial — 01. Identifying Spatial Region-Specific CCI Genes in Synthetic Data

This notebook is a **fast, self-contained walkthrough** using the bundled simulation outputs.

**Key concept:** Because the simulation includes *ground truth*, you can sanity-check the deconvolution result and then perform downstream CCI(cell-cell interaction) gene / CCI-pair discovery. 

You will:
- Load precomputed `ccisignal` outputs (reference signatures "sc_ccisignal.h5ad" + spatial deconvolution "sp_ccisignal.h5ad").
- (Optional) verify deconvolution result against ground truth proportions.
- Run the CellNeighborEX modules `ccigenes` to obtain a CCI gene list and `ccipairs` to identify interacting cell type pairs.


## Core Concept: Method Validation with Ground Truth

**This simulation assesses whether the method can accurately recover ground-truth CCI genes and their interacting cell type pairs.**

### Why Start with Simulation?

To verify that our pipeline correctly identifies CCI genes and their interacting cell type pairs, we therefore evaluate its performance on simulated data where the ground truth is explicitly defined.

**The simulation data provides the following grouth truth information:**
1. **Cell type composition** → Test deconvolution accuracy
2. **Interaction genes** → Test CCI gene discovery recall
3. **Cell type pairs** → Test interacting cell type pair precision


## Prerequisites
- Recommended working directory: `CNEXv2/run_code/` 
- Also supported: `CNEXv2/run_code/ExampleData_Tutorials/`
- The simulation example data are located at `CNEXv2/Tutorial-ExampleData/1_simulation/`.
- All outputs will be saved to `CNEXv2/run_code/tutorial_output/01_synthetic/`.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# --- Environment Configuration ---
# Set threading options to ensure stability in shared-memory OpenMP environments
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")

# --- Path Setup ---
# Define the directory for source code and project base
RUN_CODE_DIR = Path("/home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code").resolve()
BASE_DIR = RUN_CODE_DIR.parent  # Project root (CNEXv2)

# Add the base directory to sys.path to enable CellNeighborEX module imports
sys.path.insert(0, str(BASE_DIR))

# --- CellNeighborEX Module Imports ---
from CellNeighborEX.ccisignal import compute_proportion, cluster_spots_by_proportion
from CellNeighborEX.ccigenes import (
    clean_column_names,
    obtain_clean_celltype_names,
    permutation_test_all_clusters,
    chi_square_goodness_of_fit,
    compute_combined_p_value,
    simplify_gene_names,
)
from CellNeighborEX.ccipairs import (
    regress_residual_on_interaction_with_regularization,
    extract_interaction_terms,
    compare_database,
)

/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/Data_Drive_8TB/sglee/.conda/envs/cnex2_env/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settin

In [2]:
# --- Dataset and Parameter Configuration ---
DATA_DIR = BASE_DIR / "Tutorial-ExampleData" / "1_simulation"
SC_REF_FILE = DATA_DIR / "sc_ccisignal.h5ad"  # Precomputed reference signatures
VISIUM_FILE = DATA_DIR / "sp_ccisignal.h5ad"  # Precomputed deconvolution result

# Dataset-specific defaults
SPECIES = "human"
CLUSTER_INFO = "spatial_kmeans"  # Spatial region clustering for ground truth evaluation

# Tutorial execution parameters
N_PERMUTATIONS = 200
LOG_FC = 0.5
PVAL_TERM = 0.05

# --- Output Directory Setup ---
OUTPUT_DIR = RUN_CODE_DIR / "tutorial_output" / "01_synthetic"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "figures").mkdir(exist_ok=True)

# Scanpy figure settings
sc.settings.figdir = str((OUTPUT_DIR / "figures").resolve())
sc.settings.set_figure_params(dpi=120)

print(f"SC_REF_FILE : {SC_REF_FILE}")
print(f"VISIUM_FILE : {VISIUM_FILE}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

SC_REF_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/1_simulation/sc_ccisignal.h5ad
VISIUM_FILE : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/Tutorial-ExampleData/1_simulation/sp_ccisignal.h5ad
OUTPUT_DIR  : /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code/tutorial_output/01_synthetic


## 1) Load precomputed `ccisignal` outputs

The bundled example data consist of precomputed outputs from the *ccisignal* module:
- `sc_ccisignal.h5ad` contains the cell type expression signatures learned from the scRNA-seq reference data.
- `sp_ccisignal.h5ad` contains spatial deconvolution results including posterior summaries of cell type abundances for each spot.


In [3]:
# Load h5ad files
adata_ref_sig = sc.read_h5ad(SC_REF_FILE)
adata_vis = sc.read_h5ad(VISIUM_FILE)

print("Reference (sc_ccisignal):", adata_ref_sig)
print("Spatial (sp_ccisignal):  ", adata_vis)

# Verify required data structures from previous pipeline steps
assert "mod" in adata_ref_sig.uns and "mod" in adata_vis.uns, "Expected ccisignal outputs with .uns['mod']."
assert "q05_cell_abundance_w_sf" in adata_vis.obsm, "Expected deconvolution outputs in .obsm['q05_cell_abundance_w_sf']."

# Ensure 'sample' column for Scanpy compatibility
if "sample" not in adata_vis.obs.columns:
    adata_vis.obs["sample"] = "sample0"

display(adata_vis.obs.head())

Reference (sc_ccisignal): AnnData object with n_obs × n_vars = 10338 × 663
    obs: 'batch', 'celltype', '_scvi_batch', 'labels_scanvi', '_scvi_labels', 'predictions', 'leiden', '_indices'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'n_cells', 'nonz_mean'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'hvg', 'leiden', 'log1p', 'mod', 'neighbors', 'predictions_colors', 'umap'
    obsm: 'X_scANVI', 'X_umap'
    varm: 'means_per_cluster_mu_fg', 'q05_per_cluster_mu_fg', 'q95_per_cluster_mu_fg', 'stds_per_cluster_mu_fg'
    layers: 'counts'
    obsp: 'connectivities', 'distances'
Spatial (sp_ccisignal):   AnnData object with n_obs × n_vars = 2124 × 657
    obs: 'in_tissue', 'array_row', 'array_col', 'sample', '_indices', '_scvi_batch', '_scvi_labels', 'Endothelial cells', 'Epithelial cells', 'Fibroblasts', 'Macrophages', 'Plasma cells', 'T cells', 'Tumour', 'leiden', 'proportion_leiden', 'n_genes_by_counts', 'total_counts', 'mult_factor', 'clust

,in_tissue,array_row,array_col,sample,_indices,_scvi_batch,_scvi_labels,Endothelial cells,Epithelial cells,Fibroblasts,...,Tumour,leiden,proportion_leiden,n_genes_by_counts,total_counts,mult_factor,cluster,x,y,spatial_kmeans
AAACAAGTATCTCCCA-1,1,50,102,library_id,0,0,0,0.644828,5.387355,2.162262,...,0.213960,5,5,125,263.0,1.525001,Others,12378,10782,4
AAACAGAGCGACTCCT-1,1,14,94,library_id,1,0,0,0.616228,0.192573,1.744536,...,0.074896,6,6,305,3228.0,1.331846,P5.1,11584,4292,1
AAACAGTGTTCCTGGG-1,1,73,43,library_id,2,0,0,0.930763,1.619337,3.495357,...,0.683817,5,5,186,811.0,1.126752,Others,6247,14894,0
AAACCGTTCGTCCAGG-1,1,52,42,library_id,3,0,0,1.784706,0.164438,0.202913,...,0.204798,0,0,173,938.0,1.181289,Others,6164,11110,3
AAACGAGACGGTTGAT-1,1,35,79,library_id,4,0,0,0.687579,0.149830,0.176425,...,0.154591,3,3,337,5172.0,1.185978,Others,10011,8067,1


## 2) (Optional) Deconvolution sanity-check using ground truth

The simulation dataset contains ground-truth proportions in `adata_vis.uns['ground_truth_proportion']`.
We compare those to the inferred proportions from our pipeline.


In [4]:
# Compute inferred cell type proportions per spot
adata_vis = compute_proportion(adata_vis)

# Evaluate deconvolution accuracy against ground truth (if available)
if "ground_truth_proportion" not in adata_vis.uns:
    print("No ground_truth_proportion found in adata_vis.uns; skipping.")
else:
    gt = adata_vis.uns["ground_truth_proportion"]
    cell_types = list(adata_vis.uns["mod"]["factor_names"])

    gt_df = gt.copy() if isinstance(gt, pd.DataFrame) else pd.DataFrame(gt, index=adata_vis.obs_names, columns=cell_types)
    pred_df = adata_vis.obsm["proportion"].copy().loc[gt_df.index, gt_df.columns]

    # Calculate Pearson correlation per cell type
    corr = {ct: float(np.corrcoef(gt_df[ct].values, pred_df[ct].values)[0, 1]) for ct in cell_types}
    corr_s = pd.Series(corr).sort_values(ascending=False)
    print("Per-cell-type Pearson correlation (ground truth vs inferred):")
    display(corr_s.to_frame("pearson_r"))
    print("\nMedian r:", float(corr_s.median()))

Per-cell-type Pearson correlation (ground truth vs inferred):


,pearson_r
Plasma cells,0.997111
Macrophages,0.995779
Endothelial cells,0.989143
Epithelial cells,0.988612
Fibroblasts,0.987098
T cells,0.982386
Tumour,0.979596



Median r: 0.9886124435139212


## 3) Choose a niche clustering

For this simulation tutorial, the CLUSTER_INFO parameter is set to `spatial_kmeans` by default As an alternative, you may perform niche clustering using Leiden clustering based on the deconvolution results (i.e., `proportion_leiden`).


In [5]:
# Generate clusters based on cell type proportions
adata_vis = cluster_spots_by_proportion(adata_vis, n_neighbors=7, resolution=0.3)

# Verify if the target cluster information exists in metadata
print("Available cluster columns:")
cands = [c for c in ["spatial_kmeans", "proportion_leiden", "cluster", "leiden"] if c in adata_vis.obs.columns]
print(cands)

if CLUSTER_INFO not in adata_vis.obs.columns:
    raise ValueError(f"CLUSTER_INFO='{CLUSTER_INFO}' not in adata_vis.obs. Candidates: {cands}")

print("Cluster sizes:")
display(adata_vis.obs[CLUSTER_INFO].value_counts())

Available cluster columns:
['spatial_kmeans', 'proportion_leiden', 'cluster', 'leiden']
Cluster sizes:


spatial_kmeans
4    488
0    428
3    405
1    402
2    401
Name: count, dtype: int64

## 4) Build observed and expected expression matrices

CellNeighborEX v2 uses **observed expression** (spot-level counts) and **expected expression** to identify CCI genes.
Expected expression is estimated as a linear combination of scRNA-seq–derived cell-type expression signatures weighted by spatially inferred cell-type abundances, with additional gene- and spot-specific technical adjustments.


In [6]:
cluster_info = CLUSTER_INFO

# Standardize gene names
simplify_gene_names(adata_ref_sig, SPECIES)
simplify_gene_names(adata_vis, SPECIES)

# Extract signatures (gene × celltype)
factor_names = list(adata_ref_sig.uns["mod"]["factor_names"])
inf_raw = adata_ref_sig.varm.get("means_per_cluster_mu_fg", None)
if inf_raw is None:
    inf_aver2 = adata_ref_sig.obs[factor_names].copy()
elif isinstance(inf_raw, pd.DataFrame):
    inf_aver2 = inf_raw.copy()
else:
    inf_aver2 = pd.DataFrame(inf_raw, index=adata_ref_sig.var_names, columns=factor_names)

# Strip prefix from signature columns if present
prefix = "means_per_cluster_mu_fg_"
if all(isinstance(c, str) and c.startswith(prefix) for c in inf_aver2.columns):
    inf_aver2.columns = [c[len(prefix):] for c in inf_aver2.columns]

# Align genes and cell types between spatial and reference data
genes = np.intersect1d(adata_vis.var_names, inf_aver2.index)
adata_vis = adata_vis[:, genes].copy()
inf_aver2 = inf_aver2.loc[genes]

vis_factors = list(adata_vis.uns["mod"]["factor_names"])
common = [ct for ct in vis_factors if ct in inf_aver2.columns]

if not common:
    raise ValueError(f"No overlapping celltypes found.")

# ensure abundance cols exist in obs
if not all(ct in adata_vis.obs.columns for ct in common):
    adata_vis.obs[vis_factors] = pd.DataFrame(
        adata_vis.obsm["q05_cell_abundance_w_sf"], index=adata_vis.obs_names, columns=vis_factors
    )

# Compute expected expression matrices
post = adata_vis.uns["mod"]["post_sample_means"]
total_df = adata_vis.obs[common] @ inf_aver2[common].T
final_df = (total_df.mul(post["m_g"], axis=1) + np.asarray(post["s_g_gene_add"])[0]).mul(post["detection_y_s"], axis=0)

# Merge observed and expected datasets for downstream testing
meta_data = adata_vis.obs.copy()
# Create paired matrices for Observed and Expected expression values
observed_expression = pd.concat([adata_vis.to_df(), meta_data], axis=1)
expected_expression = pd.concat([final_df, meta_data], axis=1)

# Annotate conditions and modify indices to distinguish the two groups
observed_expression["condition"] = "observed"
expected_expression["condition"] = "expected"
observed_expression.index = [f"{i}_obs" for i in observed_expression.index]
expected_expression.index = [f"{i}_exp" for i in expected_expression.index]

# Concatenate datasets and build a new AnnData object for differential testing
combined_df = pd.concat([observed_expression, expected_expression])
drop_cols = list(meta_data.columns) + ["condition"]
adata_tests = sc.AnnData(X=combined_df.drop(columns=drop_cols))
adata_tests.obs["condition"] = combined_df["condition"].values
adata_tests.obs[cluster_info] = combined_df[cluster_info].astype("category")

# Final preparation of dataframes for ccigenes modules
observed_expression = clean_column_names(observed_expression)
expected_expression = clean_column_names(expected_expression)
meta_data = clean_column_names(meta_data)
inf_aver2 = clean_column_names(inf_aver2)
cell_types = obtain_clean_celltype_names(adata_vis)

print("Alignment OK:", observed_expression.shape, expected_expression.shape)

Alignment OK: (2124, 681) (2124, 681)


## 5) Detect spatial region-specific CCI genes

CCI genes are identified using a combination of a permutation test across spatial regions and a chi-square test within each region.  
Their statistical significances are integrated using the Cauchy combination test, and genes are filtered based on adjusted p-values and log fold-change.


In [8]:
# Identify genes significantly deviating from expected expression using permutation tests
perm_results = permutation_test_all_clusters(
    adata_tests,
    cluster_info=cluster_info,
    observed_expression=observed_expression,
    expected_expression=expected_expression,
    n_permutations=N_PERMUTATIONS,
    use_zeros=True,
    random_seed=42,
    path=str(OUTPUT_DIR),
)

# Perform Chi-square goodness-of-fit test for statistical refinement
chi_results = chi_square_goodness_of_fit(
    adata_tests,
    cluster_info=cluster_info,
    groupby="condition",
    reference="observed",
    target="expected",
    use_zeros=True,
)

# Combine statistical results and filter for significant CCI genes
stats_df = pd.merge(perm_results, chi_results, on=["gene", "cluster"], how="inner")
stats_df["combined_p_value_adj"] = stats_df.apply(compute_combined_p_value, axis=1)
stats_df.to_csv(OUTPUT_DIR / "allgenes_results.csv", index=False)

# Apply filters: p-value < 0.01, mean_obs > mean_exp, and LogFC threshold
gene_filter = (stats_df["combined_p_value_adj"] < 0.01) & (stats_df["mean_ref"] > stats_df["mean_tgt"]) & (stats_df["logfc"] > LOG_FC)
final_results = stats_df.loc[gene_filter].copy()
final_results.to_csv(OUTPUT_DIR / "ccigenes_results.csv", index=False)

print(f"Niche gene hits: {final_results.shape[0]}")
display(final_results.head(15))

Permutation Test Progress for Cluster 4: 100%|██████████| 657/657 [00:02<00:00, 242.96it/s]


Performing Peason's Chi-Square Test for cluster 0...


Chi-Square Test Progress: 100%|██████████| 657/657 [00:00<00:00, 1027.71it/s]


Performing Peason's Chi-Square Test for cluster 1...


Chi-Square Test Progress: 100%|██████████| 657/657 [00:00<00:00, 1118.77it/s]


...

Chi-Square Test Progress: 100%|██████████| 657/657 [00:00<00:00, 1132.77it/s]


Performing Peason's Chi-Square Test for cluster 4...


Chi-Square Test Progress: 100%|██████████| 657/657 [00:00<00:00, 1046.84it/s]


Niche gene hits: 92


,gene,cluster,perm_p_value,perm_p_value_adj,chi_stat,chi_p_value,mean_ref,mean_tgt,logfc,n_spots(observed > 0),n_spots(%),chi_p_value_adj,combined_p_value_adj
36,B2M,0,4.940656e-324,1.679823e-322,446691.909842,4.940656e-324,391.806075,243.574989,0.683540,428,50.0,3.409053e-322,2.000000e-300
43,BECN1,0,4.940656e-324,1.679823e-322,1141.412269,1.760237e-66,2.609813,0.873799,0.945958,428,50.0,7.086670e-65,2.000000e-299
75,CCND1,0,4.940656e-324,1.679823e-322,5131.856115,4.940656e-324,4.413551,1.609467,1.052820,428,50.0,3.409053e-322,2.000000e-300
96,CD40,0,4.940656e-324,1.679823e-322,2530.615646,8.732000e-295,3.768692,1.348870,1.021627,428,50.0,5.759404e-293,2.000000e-299
123,CENPF,0,4.940656e-324,1.679823e-322,605.151699,2.797936e-08,1.011682,0.221524,0.719721,428,50.0,8.420971e-07,2.000000e-299
133,CLEC4A,0,4.940656e-324,1.679823e-322,1414.041207,5.819758e-106,1.464953,0.416642,0.799085,428,50.0,2.577322e-104,2.000000e-299
161,CSF1,0,4.940656e-324,1.679823e-322,1105.326577,1.333867e-61,2.383178,0.666183,1.021832,428,50.0,5.168734e-60,2.000000e-299
191,DDR2,0,4.940656e-324,1.679823e-322,1036.977766,1.242213e-52,2.649533,0.852553,0.978197,428,50.0,4.639592e-51,2.000000e-299
231,FKBP11,0,4.940656e-324,1.679823e-322,19755.960632,4.940656e-324,14.294393,7.382257,0.867592,428,50.0,3.409053e-322,2.000000e-300
294,IER3,0,4.940656e-324,1.679823e-322,33065.863545,4.940656e-324,14.553738,7.775906,0.825641,428,50.0,3.409053e-322,2.000000e-300


## 6) Identify interacting cell type pairs for each CCI gene (`ccipairs` module)

For each detected CCI gene, we identify interacting celltype pairs using a `regression-based framework` (i.e., two-step modeling strategy).  
Specifically, residual expression signals are regressed on cell type interaction terms to estimate pairwise interaction effects. Significant interaction terms are then annotated by mapping them to curated ligand–receptor databases (`Omnipath`, `CellChat`, `CellTalk`).

In [9]:
if final_results.empty:
    print("No niche genes detected; skipping CCI pair modeling.")
else:
    # Prepare abundance data per cluster
    norm_deconv = meta_data.loc[:, cell_types].div(meta_data.loc[:, cell_types].sum(axis=1), axis=0)
    norm_deconv[cluster_info] = meta_data[cluster_info]
    cluster_summary = norm_deconv.groupby(cluster_info).mean()

    # Identify interacting cell type pairs using regularized regression
    models_per_gene, gene_analysis = regress_residual_on_interaction_with_regularization(
        observed_expression=observed_expression,
        expected_expression=expected_expression,
        celltypes=cell_types,
        cell_sig=inf_aver2,
        niche_gene_results=final_results.drop_duplicates(subset=["gene"]),
        cluster_summary=cluster_summary,
        cluster_info=cluster_info,
        alpha=0.5,
    )

    # Extract interaction terms and validate against interaction databases
    interaction_terms = extract_interaction_terms(models_per_gene, gene_analysis, p_value_threshold=PVAL_TERM)
    
    if not interaction_terms.empty:
        interaction_terms = compare_database(interaction_terms, species=SPECIES)
        interaction_terms.to_csv(OUTPUT_DIR / "ccipairs_results.csv", index=False)
        print(f"CCI pair results saved to: {OUTPUT_DIR / 'ccipairs_results.csv'}")
        display(interaction_terms.head(20))

Processing Genes: 100%|██████████| 91/91 [00:21<00:00,  4.16gene/s]


Skipping KRT5 in cluster 1 due to missing models.


Processing Queries: 100%|██████████| 694/694 [00:27<00:00, 25.11it/s]

CCI pair results saved to: /home/Data_Drive_8TB/sglee/Project/01.Neighbor/drive-download-20251001T080437Z-1-001/CNEXv2/run_code/tutorial_output/01_synthetic/ccipairs_results.csv


,gene,cluster,term,index_celltype,neighboring_celltype,coef,std_err,p_value,source_omnipath,pair_omnipath,source_cellchat,pair_cellchat,source_celltalk,pair_celltalk
19,B2M,0,Endothelial_cells_Macrophages,Endothelial_cells,Macrophages,0.184624,0.034511,8.806265e-08,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
16,B2M,0,Epithelial_cells_Macrophages,Epithelial_cells,Macrophages,0.154394,0.036332,2.142472e-05,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
9,B2M,0,Endothelial_cells_Epithelial_cells,Epithelial_cells,Endothelial_cells,0.111072,0.027013,3.925385e-05,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
15,B2M,0,Endothelial_cells_T_cells,Endothelial_cells,T_cells,0.144362,0.038015,1.461620e-04,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
14,B2M,0,Endothelial_cells_Plasma_cells,Endothelial_cells,Plasma_cells,0.109781,0.030577,3.302268e-04,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
5,B2M,0,Fibroblasts_Macrophages,Fibroblasts,Macrophages,0.097685,0.027388,3.614971e-04,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
4,B2M,0,Endothelial_cells_Fibroblasts,Fibroblasts,Endothelial_cells,0.095821,0.027127,4.119507e-04,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
2,B2M,0,Epithelial_cells_Fibroblasts,Fibroblasts,Epithelial_cells,0.090594,0.027424,9.548519e-04,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
11,B2M,0,Macrophages_Plasma_cells,Macrophages,Plasma_cells,0.110334,0.034493,1.380119e-03,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...
6,B2M,0,Epithelial_cells_Plasma_cells,Epithelial_cells,Plasma_cells,0.077454,0.025069,2.003523e-03,omnipath,single B2M-CANX; single B2M-CALR; single B2M-P...,unknown,unknown,celltalk,single B2M-HLA-F; single B2M-CD1A; single B2M-...


## Wrap-up: output files of CellNeighborEX v2
- `allgenes_results.csv`: statistics for all genes in each spatial region before filtering.
- `ccigenes_results.csv`: statistically significant CCI genes.
- `ccipairs_results.csv`: interacting cell type pairs associated with CCI genes, with database annotations.
